## Dallas Food Inspections — Data Profiling & Quality Checks (DQX)

### Objective
Profile and cleanse the Dallas Restaurant and Food Establishment Inspections dataset using Databricks Labs DQX. This notebook mirrors the Chicago profiling notebook but addresses Dallas-specific schema differences and assignment rules.

### DQX Library Modules
- **WorkspaceClient** — Authenticate with Databricks workspace
- **DQProfiler** — Profile data structure (datatypes, cardinality, unique values, stats)
- **DQGenerator** — Auto-generate data quality check rules from the profile
- **DQEngine** — Execute quality checks and split data into valid / quarantined sets

### Dataset
- **Source:** [Dallas OpenData — Restaurant and Food Establishment Inspections](https://www.dallasopendata.com/Services/Restaurant-and-Food-Establishment-Inspections-Octo/dri5-wcct)
- **Rows:** ~79K inspection records (Oct 2016 – Jan 2024)
- **Columns:** 114 (11 core + 25 violations × 4 fields + Inspection Month/Year + Lat Long)

### Assignment Requirements Covered
1. Restaurant Name, Inspection Date & Type, Zip codes cannot be null
2. Zip codes must be in valid format
3. Violation score (Inspection_Score) in Dallas cannot exceed 100
4. If violation score >= 90, cannot have more than 3 violations
5. Inspection result cannot be PASS if any violation contains Urgent/Critical terms
6. Every inspection must have at least 1 unique violation; duplicate violations loaded as distinct
7. Standardize violation schema to match Chicago (common dim_violation structure)
8. Profiling and cleansing steps documented


In [0]:
%pip install databricks-labs-dqx

In [0]:
# Restart kernel to pick up newly installed library
%restart_python

### Step 1 — Load Data from Bronze Zone
Read the Dallas Food Inspections data from the Bronze Delta table (loaded by `01_Raw_Ingestion` notebook).
> **Table:** `chicago_dallas_food_inspection.bronze.dallas_raw`

In [0]:
# Load Dallas Food Inspections from Bronze Delta table
# (Raw CSV was ingested into Bronze by Notebook 01_Raw_Ingestion)
CATALOG = "chicago_dallas_food_inspection"

df_dallas_raw = spark.table(f"{CATALOG}.bronze.dallas_raw")

# Drop ingestion metadata columns (added during raw ingestion)
if "_ingestion_timestamp" in df_dallas_raw.columns:
    df_dallas_raw = df_dallas_raw.drop("_ingestion_timestamp", "_source_city", "_source_file")

print(f"Total rows loaded from Bronze: {df_dallas_raw.count()}")
print(f"Total columns: {len(df_dallas_raw.columns)}")

In [0]:
# Preview raw data
display(df_dallas_raw)

In [0]:
# Built-in Databricks data summary
dbutils.data.summarize(df_dallas_raw)

In [0]:
# Inspect the raw schema — note the 25 repeating violation column groups
df_dallas_raw.printSchema()

### Step 2 — Type Casting & Cleanup
Columns are already renamed in the Bronze layer (by `01_Raw_Ingestion`). Apply type casting, parse Lat/Long, and trim whitespace.

In [0]:
from pyspark.sql.functions import col, trim, to_date, regexp_extract, when, lit, upper, coalesce
from pyspark.sql.types import StringType

# Bronze columns are already renamed (by 01_Raw_Ingestion)
# Just need type casting and cleanup
df_dallas = df_dallas_raw

# Cast Inspection_Date from string to date
df_dallas = df_dallas.withColumn(
    "Inspection_Date",
    to_date(col("Inspection_Date"), "MM/dd/yyyy")
)

# Cast Inspection_Score to integer
df_dallas = df_dallas.withColumn("Inspection_Score", col("Inspection_Score").cast("int"))

# Parse Lat_Long_Location "(lat, lon)" into separate Latitude/Longitude (double)
df_dallas = (
    df_dallas
    .withColumn("Latitude",
        regexp_extract(col("Lat_Long_Location"), r"\(([\-]?[\d.]+),\s*([\-]?[\d.]+)\)", 1).cast("double")
    )
    .withColumn("Longitude",
        regexp_extract(col("Lat_Long_Location"), r"\(([\-]?[\d.]+),\s*([\-]?[\d.]+)\)", 2).cast("double")
    )
)

# Cast all Violation_Points columns to integer
for v in range(1, 26):
    df_dallas = df_dallas.withColumn(f"Violation_Points_{v}", col(f"Violation_Points_{v}").cast("int"))

# Trim whitespace from all string columns
for field in df_dallas.schema.fields:
    if isinstance(field.dataType, StringType):
        df_dallas = df_dallas.withColumn(field.name, trim(col(field.name)))

print(f"Cleaned columns ({len(df_dallas.columns)})")
display(df_dallas.select(
    "Restaurant_Name", "Inspection_Type", "Inspection_Date", "Inspection_Score",
    "Street_Address", "Zip_Code", "Latitude", "Longitude"
).limit(10))

In [0]:
# Verify cleaned schema
df_dallas.printSchema()

### Step 3 — Compute Violation Count & Explode Violations
1. Count how many violations each inspection has (needed for score >= 90 rule)
2. Explode the 25 repeating column groups into individual rows — standardizing the schema to match Chicago
3. Deduplicate violations within each inspection (assignment: "duplicate violations loaded as distinct")

In [0]:
from pyspark.sql.functions import sum as _sum, array, struct, explode, size, array_distinct

# Count non-null violation descriptions per row
violation_desc_cols = [f"Violation_Description_{v}" for v in range(1, 26)]

df_dallas = df_dallas.withColumn(
    "Violation_Count",
    sum([when(col(c).isNotNull() & (col(c) != ""), lit(1)).otherwise(lit(0)) for c in violation_desc_cols])
)

print("Violation count distribution:")
display(
    df_dallas.groupBy("Violation_Count").count().orderBy("Violation_Count")
)

In [0]:
from pyspark.sql.functions import explode, array, struct, col, lit, when

# Build an array of structs for each violation slot (1-25)
violation_structs = []
for v in range(1, 26):
    violation_structs.append(
        when(
            col(f"Violation_Description_{v}").isNotNull() & (col(f"Violation_Description_{v}") != ""),
            struct(
                col(f"Violation_Description_{v}").alias("Violation_Description"),
                col(f"Violation_Points_{v}").alias("Violation_Points"),
                col(f"Violation_Detail_{v}").alias("Violation_Detail"),
                col(f"Violation_Memo_{v}").alias("Violation_Memo")
            )
        )
    )

# Combine into array, filter nulls, then explode
from pyspark.sql.functions import array, expr

df_dallas = df_dallas.withColumn(
    "Violations_Struct_Array",
    array(*violation_structs)
)

# Explode into individual violation rows
df_violations_dallas = (
    df_dallas
    .select(
        "Restaurant_Name", "Inspection_Type", "Inspection_Date", "Inspection_Score",
        "Street_Address", "Zip_Code", "Latitude", "Longitude",
        explode(col("Violations_Struct_Array")).alias("violation")
    )
    .filter(col("violation").isNotNull())
    .select(
        "Restaurant_Name", "Inspection_Date", "Inspection_Score",
        "Street_Address", "Zip_Code",
        col("violation.Violation_Description").alias("Violation_Description"),
        col("violation.Violation_Points").alias("Violation_Points"),
        col("violation.Violation_Detail").alias("Violation_Detail"),
        col("violation.Violation_Memo").alias("Violation_Comments"),  # Standardized name
        lit("Dallas").alias("City")
    )
)

# Extract violation code from description (format: "*CODE description")
df_violations_dallas = df_violations_dallas.withColumn(
    "Violation_Code",
    regexp_extract(col("Violation_Description"), r"^\*?(\d+)", 1).cast("int")
)

# Deduplicate violations within each inspection
df_violations_dallas = df_violations_dallas.dropDuplicates([
    "Restaurant_Name", "Inspection_Date", "Violation_Description"
])

print(f"Total violation rows (exploded & deduped): {df_violations_dallas.count()}")
print("\nSample parsed violations:")
display(
    df_violations_dallas
    .select("Restaurant_Name", "Inspection_Date", "Violation_Code", 
            "Violation_Description", "Violation_Points", "Violation_Comments")
    .limit(20)
)

### Step 4 — Flag Urgent/Critical Violations
Per the assignment: "Inspection result cannot be PASS if any violation contains Urgent/Critical terms."
In Dallas, a high score (>= 90) is effectively a PASS. We flag inspections that have Urgent/Critical violations.

In [0]:
# Check all 25 violation description columns for Urgent/Critical terms
urgent_critical_expr_parts = []
for v in range(1, 26):
    c = f"Violation_Description_{v}"
    urgent_critical_expr_parts.append(
        f"UPPER(COALESCE({c}, '')) LIKE '%URGENT%' OR UPPER(COALESCE({c}, '')) LIKE '%CRITICAL%'"
    )

urgent_critical_expr = " OR ".join(urgent_critical_expr_parts)

df_dallas = df_dallas.withColumn(
    "Has_Urgent_Critical",
    expr(f"CASE WHEN ({urgent_critical_expr}) THEN true ELSE false END")
)

print(f"Inspections with Urgent/Critical violations: {df_dallas.filter('Has_Urgent_Critical = true').count()}")
print(f"Of those, with score >= 90 (PASS): {df_dallas.filter('Has_Urgent_Critical = true AND Inspection_Score >= 90').count()}")

### Step 5 — DQX Profiling

In [0]:
from databricks.labs.dqx.profiler.profiler import DQProfiler
from databricks.labs.dqx.profiler.generator import DQGenerator
from databricks.labs.dqx.profiler.dlt_generator import DQDltGenerator
from databricks.labs.dqx.engine import DQEngine
from databricks.sdk import WorkspaceClient

ws = WorkspaceClient()

In [0]:
import json

profile_options = {
    "round": True,
    "max_in_count": 15,
    "distinct_ratio": 0.05,
    "max_null_ratio": 0.05,
    "remove_outliers": True,
    "outlier_columns": ["Inspection_Score", "Latitude", "Longitude"],
    "num_sigmas": 3,
    "trim_strings": True,
    "max_empty_ratio": 0.02,
    "sample_fraction": 0.3,
    "sample_seed": 42,
    "limit": 10000,
}

# Profile core columns (not all 100 violation columns)
columns_to_profile = [
    "Restaurant_Name", "Inspection_Type", "Inspection_Date", "Inspection_Score",
    "Street_Number", "Street_Name", "Street_Direction", "Street_Type",
    "Street_Unit", "Street_Address", "Zip_Code",
    "Inspection_Month", "Inspection_Year",
    "Latitude", "Longitude", "Violation_Count"
]

profiler = DQProfiler(ws)
summary_stats, profiles = profiler.profile(
    df_dallas,
    options=profile_options,
    columns=columns_to_profile
)

print("GENERATED PROFILE RULES")
print("=" * 80)
for pf in profiles:
    print(pf)

print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)
json_formatted = json.dumps(summary_stats, indent=4, default=str)
print(json_formatted)

### Step 6 — Auto-Generate DQ Rules from Profile

In [0]:
generator = DQGenerator(ws)
auto_checks = generator.generate_dq_rules(profiles)

print("AUTO-GENERATED DQ RULES")
print("=" * 80)
for chk in auto_checks:
    print(chk)

In [0]:
dqengine = DQEngine(ws)
auto_results = dqengine.apply_checks_by_metadata(df_dallas, auto_checks)
display(auto_results)

### Step 7 — Duplicate Detection

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import count

# Duplicate detection: same restaurant + date + type + zip
dup_key_cols = ["Restaurant_Name", "Inspection_Date", "Inspection_Type", "Zip_Code"]
w = Window.partitionBy(*dup_key_cols)
df_dallas_qc = df_dallas.withColumn(
    "dup_count",
    count("*").over(w)
)

print(f"Rows with duplicates: {df_dallas_qc.filter('dup_count > 1').count()}")

### Step 8 — Custom Data Quality Rules (Assignment Requirements)
Expectations applied as DQX checks — bad rows are dropped:
1. **Restaurant Name** cannot be null
2. **Inspection Date** cannot be null
3. **Inspection Type** cannot be null
4. **Zip code** cannot be null and must be valid format
5. **Violation score** cannot exceed 100
6. **If score >= 90**, cannot have more than 3 violations
7. **Cannot be PASS** (score >= 90) if any violation contains Urgent/Critical
8. **Every inspection must have at least 1 unique violation**
9. No future dates, no duplicates


In [0]:
import yaml
from pyspark.sql.functions import expr

custom_checks = yaml.safe_load("""

# ── MANDATORY FIELD CHECKS ──

- criticality: error
  check:
    function: sql_expression
    arguments:
      expression: "Restaurant_Name IS NOT NULL AND TRIM(Restaurant_Name) != ''"
      msg: "Restaurant_Name cannot be null or empty"

- criticality: error
  check:
    function: sql_expression
    arguments:
      expression: "Inspection_Date IS NOT NULL"
      msg: "Inspection_Date cannot be null"

- criticality: error
  check:
    function: sql_expression
    arguments:
      expression: "Inspection_Type IS NOT NULL AND TRIM(Inspection_Type) != ''"
      msg: "Inspection_Type cannot be null or empty"

# ── ZIP CODE VALIDATION ──

- criticality: error
  check:
    function: sql_expression
    arguments:
      expression: "Zip_Code IS NOT NULL AND TRIM(Zip_Code) != ''"
      msg: "Zip_Code cannot be null or empty"

- criticality: error
  check:
    function: sql_expression
    arguments:
      expression: "Zip_Code IS NULL OR TRIM(Zip_Code) = '' OR Zip_Code RLIKE '^[0-9]{5}(-[0-9]{4})?$'"
      msg: "Zip_Code must be in valid US format (5 digits or 5+4)"

# ── VIOLATION SCORE CANNOT EXCEED 100 (Dallas-specific) ──

- criticality: error
  check:
    function: sql_expression
    arguments:
      expression: "Inspection_Score <= 100"
      msg: "Dallas violation score cannot exceed 100"

- criticality: warn
  check:
    function: sql_expression
    arguments:
      expression: "Inspection_Score >= 0"
      msg: "Inspection_Score is negative — possible data entry error"

# ── SCORE >= 90 CANNOT HAVE > 3 VIOLATIONS (Dallas-specific) ──

- criticality: error
  check:
    function: sql_expression
    arguments:
      expression: "NOT (Inspection_Score >= 90 AND Violation_Count > 3)"
      msg: "If violation score >= 90, cannot have more than 3 violations"

# ── CANNOT BE PASS IF VIOLATION CONTAINS URGENT/CRITICAL (Dallas-specific) ──

- criticality: error
  check:
    function: sql_expression
    arguments:
      expression: "NOT (Inspection_Score >= 90 AND Has_Urgent_Critical = true)"
      msg: "Inspection cannot be PASS (score >= 90) if any violation contains Urgent/Critical"

# ── EVERY INSPECTION MUST HAVE >= 1 UNIQUE VIOLATION ──

- criticality: error
  check:
    function: sql_expression
    arguments:
      expression: "Violation_Count >= 1"
      msg: "Every inspection must have at least 1 unique violation"

# ── INSPECTION TYPE DOMAIN ──

- criticality: warn
  check:
    function: sql_expression
    arguments:
      expression: "Inspection_Type IN ('Routine', 'Complaint', 'Follow-up')"
      msg: "Unexpected Inspection_Type value"

# ── DATE VALIDATION ──

- criticality: error
  check:
    function: sql_expression
    arguments:
      expression: "Inspection_Date IS NULL OR Inspection_Date <= current_date()"
      msg: "Inspection_Date cannot be in the future"

# ── GEO COORDINATE VALIDATION (Dallas metro) ──

- criticality: warn
  check:
    function: sql_expression
    arguments:
      expression: "Latitude IS NULL OR (Latitude BETWEEN 32.0 AND 34.0)"
      msg: "Latitude outside Dallas metro range"

- criticality: warn
  check:
    function: sql_expression
    arguments:
      expression: "Longitude IS NULL OR (Longitude BETWEEN -98.0 AND -96.0)"
      msg: "Longitude outside Dallas metro range"

# ── DUPLICATE DETECTION ──

- criticality: error
  check:
    function: sql_expression
    arguments:
      expression: "dup_count = 1"
      msg: "Duplicate inspection record detected"

""")

### Step 9 — Run Custom Checks & Split Valid / Quarantine

In [0]:
dqengine = DQEngine(ws)

valid, quarantine = dqengine.apply_checks_by_metadata_and_split(
    df_dallas_qc,
    custom_checks,
    globals()
)

print(f"Total rows:      {df_dallas_qc.count()}")
print(f"Valid rows:       {valid.count()}")
print(f"Quarantine rows:  {quarantine.count()}")

In [0]:
# Preview quarantined rows
display(quarantine)

In [0]:
# Preview valid rows
display(valid)

### Step 10 — Analyze Quality Issues

In [0]:
from pyspark.sql import functions as F

print("QUARANTINE BREAKDOWN BY RULE")
print("=" * 60)
display(
    quarantine
    .select(F.explode("_errors").alias("e"))
    .groupBy(F.col("e.name").alias("rule"))
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
# Detailed quarantine analysis
print(f"Duplicate rows: {quarantine.filter('dup_count > 1').count()}")
print(f"Rows with score > 100: {quarantine.filter('Inspection_Score > 100').count()}")
print(f"Rows with score >= 90 AND > 3 violations: {quarantine.filter('Inspection_Score >= 90 AND Violation_Count > 3').count()}")
print(f"Rows with 0 violations: {quarantine.filter('Violation_Count < 1').count()}")
print(f"Rows with Urgent/Critical + score >= 90: {quarantine.filter('Has_Urgent_Critical = true AND Inspection_Score >= 90').count()}")
print(f"Negative score rows: {quarantine.filter('Inspection_Score < 0').count()}")

### Step 11 — Remediation
Cleansing steps:
1. Remove rows with null Restaurant_Name, Inspection_Date, Inspection_Type, Zip_Code
2. Filter invalid zip codes
3. Cap Inspection_Score at 100, remove negatives
4. Remove rows where score >= 90 and > 3 violations
5. Remove rows where score >= 90 and has Urgent/Critical violations
6. Require at least 1 unique violation
7. Remove future dates
8. Deduplicate
9. Fill optional null fields

In [0]:
from pyspark.sql.functions import col, current_date

df_remediated = (
    df_dallas
    # 1. Remove nulls on mandatory fields
    .filter(col("Restaurant_Name").isNotNull() & (trim(col("Restaurant_Name")) != ""))
    .filter(col("Inspection_Date").isNotNull())
    .filter(col("Inspection_Type").isNotNull() & (trim(col("Inspection_Type")) != ""))
    .filter(col("Zip_Code").isNotNull() & (trim(col("Zip_Code")) != ""))
    # 2. Valid zip format
    .filter(col("Zip_Code").rlike(r"^[0-9]{5}(-[0-9]{4})?$"))
    # 3. Score validations
    .filter(col("Inspection_Score") <= 100)
    .filter(col("Inspection_Score") >= 0)
    # 4. Score >= 90 cannot have > 3 violations
    .filter(~((col("Inspection_Score") >= 90) & (col("Violation_Count") > 3)))
    # 5. Score >= 90 cannot have Urgent/Critical
    .filter(~((col("Inspection_Score") >= 90) & (col("Has_Urgent_Critical") == True)))
    # 6. At least 1 unique violation
    .filter(col("Violation_Count") >= 1)
    # 7. No future dates
    .filter(col("Inspection_Date") <= current_date())
    # 8. Deduplicate
    .dropDuplicates(["Restaurant_Name", "Inspection_Date", "Inspection_Type", "Zip_Code"])
    # 9. Fill optional fields
    .fillna({
        "Street_Direction": "Unknown",
        "Street_Type": "Unknown",
        "Street_Unit": "Unknown"
    })
)

print(f"Rows after remediation: {df_remediated.count()}")

### Step 12 — Re-Check After Remediation

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import count

df_remediated_qc = df_remediated.withColumn(
    "dup_count",
    count("*").over(Window.partitionBy("Restaurant_Name", "Inspection_Date", "Inspection_Type", "Zip_Code"))
)

valid2, quarantine2 = dqengine.apply_checks_by_metadata_and_split(
    df_remediated_qc,
    custom_checks,
    globals()
)

print("AFTER REMEDIATION:")
print(f"  Valid rows:       {valid2.count()}")
print(f"  Quarantine rows:  {quarantine2.count()}")

### Step 13 — Save to Silver Zone

In [0]:
# Prepare Silver table — drop internal/temp columns
drop_cols = ["_errors", "_warnings", "dup_count", "Violations_Struct_Array", "Lat_Long_Location", "Has_Urgent_Critical"]
silver_cols = [c for c in valid2.columns if c not in drop_cols]
df_silver = valid2.select(*silver_cols)

# Save inspection-level Silver table
df_silver.write.mode("overwrite").saveAsTable("dallas_food_inspections_silver")

# Save exploded violations table (standardized schema matching Chicago)
# Only include violations from valid inspections
df_violations_silver = (
    df_violations_dallas
    .join(
        df_silver.select("Restaurant_Name", "Inspection_Date").distinct(),
        ["Restaurant_Name", "Inspection_Date"],
        "inner"
    )
)

df_violations_silver.write.mode("overwrite").saveAsTable("dallas_violations_silver")

# Save quarantine for audit
quarantine.write.mode("overwrite").saveAsTable("dallas_food_inspections_quarantine")

print("Tables saved successfully:")
print("  - dallas_food_inspections_silver (inspection-level, cleansed)")
print("  - dallas_violations_silver (exploded violations, standardized schema)")
print("  - dallas_food_inspections_quarantine (rejected rows)")

### Step 14 — Final Data Quality Summary

In [0]:
print("=" * 70)
print("DALLAS FOOD INSPECTIONS — DATA QUALITY SUMMARY")
print("=" * 70)
print(f"Total input rows:                {df_dallas_raw.count()}")
print(f"Rows after column cleanup:       {df_dallas.count()}")
print(f"Valid rows (Silver):             {valid2.count()}")
print(f"Quarantine rows:                 {quarantine.count()}")
print(f"Total violations (exploded):     {df_violations_silver.count()}")
print("=" * 70)

print("\nSilver dataset preview:")
display(df_silver.select(
    "Restaurant_Name", "Inspection_Type", "Inspection_Date", "Inspection_Score",
    "Street_Address", "Zip_Code", "Violation_Count"
).limit(20))

print("\nExploded violations preview (standardized schema):")
display(df_violations_silver.select(
    "Restaurant_Name", "Inspection_Date", "Violation_Code",
    "Violation_Description", "Violation_Points", "Violation_Comments", "City"
).limit(20))

print("\nQuarantine dataset preview:")
display(quarantine.limit(20))

### Step 15 — Standardized Violation Schema (Dallas ↔ Chicago)
Both datasets' violations are standardized to a common schema for the dimensional model:

| Standardized Column | Dallas Source | Chicago Source |
|---|---|---|
| `Violation_Code` | Extracted from `Violation_Description_N` (e.g., `*01` → 1) | Parsed from `"CODE. DESC"` text (e.g., `53.` → 53) |
| `Violation_Description` | `Violation_Description_N` column directly | Parsed from `"CODE. DESC - Comments: ..."` |
| `Violation_Points` | `Violation_Points_N` column directly | N/A (Chicago uses derived `Violation_Score` at inspection level) |
| `Violation_Comments` | `Violation_Memo_N` column | Parsed from text after `"- Comments:"` |
| `Violation_Detail` | `Violation_Detail_N` column | N/A |
| `City` | "Dallas" (literal) | "Chicago" (literal) |

**Key notes:**
- Dallas codes (e.g., `*01 Cooling`, `*10 Chlorine sanitizer`) and Chicago codes (e.g., `53. TOILET FACILITIES`) do NOT need to match — different regulatory frameworks
- Both sets stored in a single `dim_violation` dimension in Gold with `City` column to distinguish source
- Duplicate violations within an inspection are deduplicated (loaded as distinct) per assignment


### Profiling Findings — Documentation

#### Schema Overview (Core Columns)
| Column | Type | Nulls | Notes |
|--------|------|-------|-------|
| Restaurant_Name | string | ~0% (11 nulls) | Business name |
| Inspection_Type | string | 0% | 3 distinct: Routine, Complaint, Follow-up |
| Inspection_Date | date | 0% | Oct 2016 – Jan 2024 |
| Inspection_Score | int | 0% | Range: -26 to 100. 6 rows have negative scores |
| Street_Number | string | 0% | |
| Street_Name | string | 0% | |
| Street_Direction | string | 67% null | Optional — N/S/E/W prefix |
| Street_Type | string | 2.1% null | RD, ST, AVE, etc. |
| Street_Unit | string | 64.3% null | Optional suite/unit |
| Street_Address | string | 0% | Full composite address |
| Zip_Code | string | 0% | 156 distinct — some non-Dallas zips |
| Inspection_Month | string | 0% | "Mon YYYY" format |
| Inspection_Year | string | 0% | "FYXXXX" format |
| Lat_Long_Location | string | 10.9% null | Parsed into Latitude/Longitude (double) |

#### Violation Column Structure (25 groups × 4 fields each)
| Violation # | Description Null % | Notes |
|-------------|-------------------|-------|
| 1 | 8.3% | Most inspections have at least 1 violation |
| 2 | 17.9% | |
| 3 | 29.2% | |
| 4 | 40.5% | |
| 5 | 50.9% | Over half have ≤ 4 violations |
| 6–25 | Increasing | Progressively sparser |

#### Key Quality Observations
1. **Negative Inspection_Score (6 rows):** Range -26 to -3. Data entry errors — valid range 0–100.
2. **Score >= 90 with > 3 violations (~18,301 rows):** Largest quality issue. Assignment rule violation.
3. **Urgent/Critical violations (3 rows):** Very few rows. Checked across all 25 description columns.
4. **Inspections with 0 violations (~6,579 rows):** ~8.3% of inspections have no violations recorded.
5. **Street_Direction (67% null):** Expected — not all addresses have directional prefix.
6. **Lat_Long_Location (10.9% null):** Some lack geocoding. Parsed into Lat/Long doubles.
7. **Column typo:** `Violation  Memo - 20` has double space — fixed in rename step.
8. **Inspection_Type:** Very clean — only 3 values vs Chicago's 111 messy types.

#### Cleansing Steps Applied
| Step | Action | Rows Affected |
|------|--------|--------------|
| Null Restaurant_Name | Dropped | ~11 rows |
| Null Inspection_Date | Dropped | 0 rows |
| Null Inspection_Type | Dropped | 0 rows |
| Null/invalid Zip_Code | Dropped | ~0 rows |
| Score > 100 | Dropped | 0 rows |
| Negative score | Dropped | 6 rows |
| Score >= 90 + > 3 violations | Dropped | ~18,301 rows |
| Score >= 90 + Urgent/Critical | Dropped | Check at runtime |
| 0 violations | Dropped | ~6,579 rows |
| Future dates | Dropped | 0 rows |
| Duplicates | Deduplicated | Check at runtime |
| Null Street_Direction | Filled "UNKNOWN" | ~52,898 rows |
| Null Street_Type | Filled "UNKNOWN" | ~1,674 rows |
| Null Street_Unit | Filled "UNKNOWN" | ~50,804 rows |
| Violation dedup (exploded) | dropDuplicates | At runtime |
